<a href="https://colab.research.google.com/github/OlegSamarsky/MN/blob/main/%D0%9B%D0%A09_%D0%A1%D0%B0%D0%BC%D0%B0%D1%80%D1%81%D1%8C%D0%BA%D0%B8%D0%B9_%D0%A4%D0%86%D0%A23_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторна робота № 9
## Рекомендаційні системи: SVD та KNNBasic

**ПІБ студента:** Самарський Олег Ростиславович  
**Група:** ФІТ 3-10  
**Дата:** 18.11.2026

> **Призначення:** шаблон для самостійного виконання ЛР на основі прикладу з алгоритмами `SVD` і `KNNBasic` у бібліотеці `Surprise`. Комірки з позначкою **TODO** потрібно заповнити власним кодом. Це не готовий розв’язок і не заміна офіційної умови роботи.

**Мета:** завантажити дані про оцінки фільмів, побудувати дві рекомендаційні моделі, підібрати гіперпараметри, порівняти моделі за RMSE/MAE та сформувати топ-10 рекомендацій для користувача.

**Дані:** вбудований набір `MovieLens 100K` (`ml-100k`) із бібліотеки `Surprise`.

**Послідовність роботи:** підготовка середовища → дані → розбиття → налаштування моделей → навчання → тестування → рекомендації → висновки.

## 1. Підготовка середовища

Комбінація бібліотек для **Google Colab із Python 3.13**. Встановлення без `--force-reinstall`, щоб без потреби не перевстановлювати всі залежності. Після зміни NumPy чи SciPy у вже запущеному середовищі **перезапустіть сесію** (`Runtime → Restart session`), а потім виконайте комірки нижче з початку.

In [1]:
%pip install -q \
    numpy==2.2.6 \
    scipy==1.15.3 \
    scikit-learn==1.6.1 \
    pandas==2.2.3 \
    scikit-surprise==1.1.5

In [2]:
import sys
import numpy as np
import scipy
import sklearn
import pandas as pd
import surprise

from surprise import Dataset, Reader, SVD, KNNBasic, accuracy
from surprise.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split as sklearn_train_test_split

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('SciPy:', scipy.__version__)
print('scikit-learn:', sklearn.__version__)
print('pandas:', pd.__version__)
print('Surprise:', surprise.__version__)

Python: 3.13.18
NumPy: 2.4.6
SciPy: 1.15.3
scikit-learn: 1.6.1
pandas: 2.2.3
Surprise: 1.1.5


## 2. Завантаження та первинний аналіз даних

1. Завантажте вбудований набір `ml-100k`.
2. Перетворіть оцінки у `pandas.DataFrame` з колонками `user_id`, `item_id`, `rating`, `timestamp`.
3. Виведіть розмір таблиці, декілька рядків, кількість унікальних користувачів та фільмів, розподіл оцінок.

**Зверніть увагу:** ідентифікатори користувачів і фільмів у `Surprise` є зовнішніми (raw) ID; вони можуть мати рядковий тип.

In [3]:
# Завантаження даних (готова частина шаблону)
data = Dataset.load_builtin('ml-100k', prompt=False)

ratings_df = pd.DataFrame(
    data.raw_ratings,
    columns=['user_id', 'item_id', 'rating', 'timestamp']
)

print('Розмір:', ratings_df.shape)
display(ratings_df.head())

Trying to download dataset from https://files.grouplens.org/datasets/movielens/ml-100k.zip...
Done! Dataset ml-100k has been saved to /root/.surprise_data/ml-100k
Розмір: (100000, 4)


,user_id,item_id,rating,timestamp
0,196,242,3.0,881250949
1,186,302,3.0,891717742
2,22,377,1.0,878887116
3,244,51,2.0,880606923
4,166,346,1.0,886397596


In [4]:
print("Кількість унікальних користувачів:", ratings_df['user_id'].nunique())
print("Кількість унікальних фільмів:", ratings_df['item_id'].nunique())
print("Мінімальний рейтинг:", ratings_df['rating'].min())
print("Максимальний рейтинг:", ratings_df['rating'].max())
print(f"Середній рейтинг: {ratings_df['rating'].mean():.4f}")
print("\nПропущені значення:")
print(ratings_df.isnull().sum())

Кількість унікальних користувачів: 943
Кількість унікальних фільмів: 1682
Мінімальний рейтинг: 1.0
Максимальний рейтинг: 5.0
Середній рейтинг: 3.5299

Пропущені значення:
user_id      0
item_id      0
rating       0
timestamp    0
dtype: int64


**Короткі спостереження щодо даних:**


Досліджуваний датасет `ml-100k` охоплює 100 000 оцінок, виставлених користувачами різним кінострічкам, і характеризується повною відсутністю пропусків. Рейтинги розподілені в діапазоні від `1.0` до `5.0` зі зміщенням у бік вищих балів (середнє значення становить близько `3.53`). Оскільки унікальних фільмів значно більше, ніж користувачів, сформована матриця взаємодій має високий ступінь розрідженості.

## 3. Розбиття на навчальну та тестову вибірки

Виконайте випадкове розбиття **80% / 20%** із `random_state=42` за прикладом. Для налаштування гіперпараметрів використовуйте **лише навчальну** частину. Тестова вибірка має залишатися недоторканою до фінальної оцінки.

In [5]:
train_df, test_df = sklearn_train_test_split(
    ratings_df,
    test_size=0.2,
    random_state=42
)

print('Навчальна вибірка:', train_df.shape)
print('Тестова вибірка:', test_df.shape)

Навчальна вибірка: (80000, 4)
Тестова вибірка: (20000, 4)


In [6]:
reader = Reader(
    rating_scale=data.reader.rating_scale
)

train_data = Dataset.load_from_df(
    train_df[['user_id', 'item_id', 'rating']],
    reader
)

testset = list(
    test_df[['user_id', 'item_id', 'rating']]
    .itertuples(index=False, name=None)
)

print('Розмір train_df:', len(train_df))
print('Розмір testset:', len(testset))

Розмір train_df: 80000
Розмір testset: 20000


## 4. Гіперпараметри моделей

Для `SVD` розгляньте кількість прихованих факторів, швидкість навчання та регуляризацію. Для `KNNBasic` — число сусідів, мінімальне число сусідів та метрику схожості.

Нижче — **початкова сітка** з аналізованого прикладу. Її можна змінити й обґрунтувати вибір.

In [8]:
# Стартові сітки параметрів (можна змінювати)
param_grid_svd = {
    'n_factors': [50, 100],
    'lr_all': [0.005, 0.010],
    'reg_all': [0.02, 0.05]
}

param_grid_knn = {
    'k': [20, 40],
    'min_k': [1, 5],
    'sim_options': {
        'name': ['cosine', 'pearson'],
        'user_based': [False, True]
    }
}

**Обґрунтування вибору параметрів:**


* **Алгоритм SVD:** підбір зосереджено на трьох ключових параметрах: розмірності латентного простору `n_factors` (`[50, 100]`), яка регулює деталізацію прихованих вподобань; темпі навчання `lr_all` (`[0.005, 0.010]`) для збалансованої збіжності градієнтного спуску; та параметрі L2-регуляризації `reg_all` (`[0.02, 0.05]`), що стримує перенавчання на неповних даних.
* **Алгоритм KNNBasic:** сітка параметрів охоплює розмір околиці `k` (`[20, 40]`) та мінімальну кількість спільних рейтингів `min_k` (`[1, 5]`), необхідну для надійного розрахунку подібності. Додатково в словнику `sim_options` порівнюються косинусна міра (`cosine`) з кореляцією Пірсона (`pearson`), а також підходи *user-based* (`True`) та *item-based* (`False`).

## 5. Пошук найкращих гіперпараметрів (GridSearchCV)

Для обох моделей використайте `GridSearchCV` з метриками `rmse` та `mae` і `cv=3`. Навчайте об’єкти пошуку **лише на `train_data`**.

In [9]:
grid_search_svd = GridSearchCV(
    SVD,
    param_grid_svd,
    measures=['rmse', 'mae'],
    cv=3,
    n_jobs=-1
)

grid_search_svd.fit(train_data)

In [10]:
grid_search_knn = GridSearchCV(
    KNNBasic,
    param_grid_knn,
    measures=['rmse', 'mae'],
    cv=3,
    n_jobs=-1
)

grid_search_knn.fit(train_data)

In [11]:
print('\nНайкращий результат SVD:')
print('RMSE:', grid_search_svd.best_score['rmse'])
print('MAE:', grid_search_svd.best_score['mae'])
print('Параметри за RMSE:', grid_search_svd.best_params['rmse'])

print('\nНайкращий результат KNNBasic:')
print('RMSE:', grid_search_knn.best_score['rmse'])
print('MAE:', grid_search_knn.best_score['mae'])
print('Параметри за RMSE:', grid_search_knn.best_params['rmse'])


Найкращий результат SVD:
RMSE: 0.9445879362774668
MAE: 0.7446462347748214
Параметри за RMSE: {'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05}

Найкращий результат KNNBasic:
RMSE: 1.0268375952205944
MAE: 0.8144871236389811
Параметри за RMSE: {'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}}


**Аналіз результатів кросвалідації:**

| Модель | Найкращий CV RMSE | CV MAE | Найкращі параметри за RMSE |
| :--- | :---: | :---: | :--- |
| **SVD** | `0.9442` | `0.7455` | `{'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05}` |
| **KNNBasic** | `1.0278` | `0.8155` | `{'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}}` |

Під час 3-фолдової кросвалідації алгоритм матричної факторизації **SVD** виявився точнішим за сусідський метод **KNNBasic**: середньоквадратична похибка `CV RMSE` зменшилася на `0.0836` (`0.9442` проти `1.0278`), а середня абсолютна похибка `CV MAE` — на `0.0700` (`0.7455` проти `0.8155`). Для **SVD** найкраще спрацювала модель із 50 латентними факторами (`n_factors=50`), підвищеною швидкістю навчання (`lr_all=0.01`) та сильнішою регуляризацією (`reg_all=0.05`). У випадку **KNNBasic** оптимум досягнуто при пошуку схожих користувачів (`user_based=True`) за косинусною метрикою (`cosine`) з урахуванням до 40 сусідів (`k=40`, `min_k=1`).

## 6. Навчання остаточних моделей

Побудуйте `trainset = train_data.build_full_trainset()`. Створіть нові екземпляри SVD і KNNBasic з параметрами, обраними за **CV RMSE**, після чого навчіть їх на **всій навчальній вибірці**.

In [12]:
# TODO 6.1 – Отримайте best_svd_params і best_knn_params за CV RMSE.
best_svd_params = grid_search_svd.best_params['rmse']
best_knn_params = grid_search_knn.best_params['rmse']

# TODO 6.2 – Створіть trainset із train_data.
trainset = train_data.build_full_trainset()

# TODO 6.3 – Навчіть best_svd і best_knn.
best_svd = SVD(
    **best_svd_params,
    random_state=42
)
best_svd.fit(trainset)

best_knn = KNNBasic(
    **best_knn_params
)
best_knn.fit(trainset)

Computing the cosine similarity matrix...
Done computing similarity matrix.


## 7. Оцінювання на незалежній тестовій вибірці

Отримайте передбачення для `testset` методами `.test()` обох моделей. Обчисліть `RMSE` та `MAE` за допомогою `surprise.accuracy`.

**Методична примітка:** вибирайте модель для рекомендацій за результатами **кросвалідації**, а незалежну тестову вибірку використовуйте для фінальної оцінки якості, а не для повторного підбору.

In [13]:
# TODO 7.1 – Отримайте svd_predictions та knn_predictions.
svd_predictions = best_svd.test(testset)
knn_predictions = best_knn.test(testset)

# TODO 7.2 – Обчисліть svd_rmse, svd_mae, knn_rmse, knn_mae.
print('\nРезультати SVD на тестовій вибірці:')
svd_rmse = accuracy.rmse(svd_predictions, verbose=True)
svd_mae = accuracy.mae(svd_predictions, verbose=True)

print('\nРезультати KNNBasic на тестовій вибірці:')
knn_rmse = accuracy.rmse(knn_predictions, verbose=True)
knn_mae = accuracy.mae(knn_predictions, verbose=True)


Результати SVD на тестовій вибірці:
RMSE: 0.9192
MAE:  0.7234

Результати KNNBasic на тестовій вибірці:
RMSE: 1.0120
MAE:  0.7984


In [14]:
# TODO 7.3 – Сформуйте DataFrame results із колонками:
results = pd.DataFrame({
    'Model': ['SVD', 'KNNBasic'],
    'RMSE': [svd_rmse, knn_rmse],
    'MAE': [svd_mae, knn_mae]
}).sort_values('RMSE')

print('\nПорівняння моделей:')
display(results)

best_model_name = results.iloc[0]['Model']
print(f"Найкраща модель за RMSE: {best_model_name}")


Порівняння моделей:


,Model,RMSE,MAE
0,SVD,0.919175,0.723355
1,KNNBasic,1.011953,0.798421


Найкраща модель за RMSE: SVD


**Таблиця тестових результатів:**

| Модель | Test RMSE | Test MAE |
| :--- | :---: | :---: |
| **SVD** | `0.9192` | `0.7234` |
| **KNNBasic** | `1.0120` | `0.7984` |

**Порівняння з CV:**
Показники якості на незалежному тестовому наборі (20 000 оцінок) повністю узгоджуються з результатами кросвалідації та є навіть трохи кращими: для **SVD** похибка `RMSE` знизилася з `0.9442` до `0.9192`, а для **KNNBasic** — з `1.0278` до `1.0120`. Це закономірний ефект, оскільки на етапі перехресної перевірки моделі бачили лише дві третини навчальних даних (~53.3 тис. записів), тоді як фінальне тренування відбувалося на всьому обсязі `trainset` (80 000 записів). Отримані значення підтверджують стійкість моделей та відсутність перенавчання.

## 8. Топ-10 рекомендацій для користувача

1. Оберіть модель із кращим **CV RMSE**.
2. Задайте `user_id` (вихідний приклад використовує `'196'`).
3. Отримайте фільми, які цей користувач уже оцінив у навчальній вибірці.
4. Для решти фільмів обчисліть прогнозований рейтинг `.predict(uid=..., iid=...).est`.
5. Виведіть **10 фільмів із найбільшим прогнозованим рейтингом**.

**Важливо:** не пропонуйте користувачу фільми, які він уже оцінив у навчальних даних. Для коректного перетворення внутрішніх та зовнішніх ID використовуйте `trainset.to_inner_uid()`, `trainset.to_raw_iid()`.

In [15]:
# Початковий ID можна змінити
user_id = '196'

# TODO 8.1 – Визначте найкращу за CV RMSE модель.
if grid_search_svd.best_score['rmse'] <= grid_search_knn.best_score['rmse']:
    best_model = best_svd
    selected_model_name = 'SVD'
else:
    best_model = best_knn
    selected_model_name = 'KNNBasic'

# TODO 8.2 – Перевірте, що користувач присутній у trainset.
try:
    inner_uid = trainset.to_inner_uid(user_id)
    print(f"Для рекомендацій використовується модель: {selected_model_name}")

    rated_inner_iids = {iid for (iid, _) in trainset.ur[inner_uid]}
    print(f"Кількість оцінених фільмів користувачем {user_id}: {len(rated_inner_iids)}")

    # TODO 8.3 – Знайдіть неоцінені ним фільми.
    unrated_raw_iids = [
        trainset.to_raw_iid(inner_iid)
        for inner_iid in trainset.all_items()
        if inner_iid not in rated_inner_iids
    ]

    # TODO 8.4 – Спрогнозуйте оцінки та покажіть топ-10.
    user_predictions = [
        (raw_iid, best_model.predict(uid=user_id, iid=raw_iid).est)
        for raw_iid in unrated_raw_iids
    ]

    recommendations_df = pd.DataFrame(
        user_predictions,
        columns=['item_id', 'predicted_rating']
    ).sort_values(by='predicted_rating', ascending=False).head(10).reset_index(drop=True)

    print(f"\nТоп-10 рекомендацій для користувача {user_id}:")
    display(recommendations_df)

except ValueError:
    print(
        f"Користувача з ID {user_id} "
        "немає в навчальній вибірці."
    )

Для рекомендацій використовується модель: SVD
Кількість оцінених фільмів користувачем 196: 29

Топ-10 рекомендацій для користувача 196:


,item_id,predicted_rating
0,320,4.490358
1,64,4.470680
2,480,4.413710
3,313,4.401920
4,513,4.399624
5,483,4.398092
6,318,4.392102
7,656,4.385379
8,22,4.377388
9,272,4.376010


## 9. Перевірка виконання

Позначте після реалізації:

- [ ] Дані `ml-100k` завантажено та перевірено.
- [ ] Є розбиття 80/20 з фіксованим `random_state`.
- [ ] `Reader`, `train_data` і `testset` створені коректно.
- [ ] GridSearchCV для **обох** моделей виконано лише на `train_data`.
- [ ] Для SVD і KNNBasic показано параметри та результати CV.
- [ ] Моделі навчено на повній навчальній частині.
- [ ] На незалежній тестовій вибірці розраховано **RMSE та MAE** для обох моделей.
- [ ] Наведено порівняльну таблицю та інтерпретацію.
- [ ] Сформовано топ-10 рекомендацій без уже оцінених фільмів.
- [ ] Усі комірки виконуються послідовно після перезапуску середовища.

**Висновок:**


За підсумками 3-фолдової перехресної перевірки для побудови рекомендаційної системи обрано модель **SVD**, оскільки вона забезпечила суттєво нижчу похибку прогнозування (`CV RMSE = 0.9442`, `CV MAE = 0.7455`) порівняно з алгоритмом **KNNBasic** (`CV RMSE = 1.0278`, `CV MAE = 0.8155`). Найвищу точність на кросвалідації модель **SVD** продемонструвала за конфігурації `{'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05}`, тоді як для **KNNBasic** оптимальним виявився підхід *user-based* із косинусною метрикою `{'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}}`. Оцінювання на відкладеній тестовій вибірці (20 000 записів) підтвердило перевагу матричної факторизації: підсумкові метрики для **SVD** склали `Test RMSE = 0.9192` і `Test MAE = 0.7234`, тоді як **KNNBasic** отримав `Test RMSE = 1.0120` та `Test MAE = 0.7984`. Порівняння етапів CV та тестування показує помірне зменшення похибок на тесті (на `0.0250` за RMSE для SVD та на `0.0158` для KNNBasic), що зумовлено збільшенням обсягу навчальних даних до повних 80 000 записів при фінальному тренуванні й свідчить про відсутність перенавчання. За допомогою навченої моделі **SVD** для користувача з ідентифікатором `196` (у профілі якого зафіксовано 29 оцінених фільмів) сформовано список із 10 найкращих рекомендацій із прогнозованими оцінками в інтервалі `4.38–4.49` бала. Водночас запропонований підхід колаборативної фільтрації вразливий до проблеми «холодного старту»: за відсутності історії взаємодій для нових користувачів або щойно доданих фільмів алгоритм не може обчислити латентні фактори чи сусідів і змушений повертати усереднений рейтинг по вибірці.

**Додаткове завдання. Написати власний рекомендаційний алгоритм з нуля**



In [16]:
import numpy as np
import pandas as pd


class BiasedSVDFromScratch:
    def __init__(self, n_factors=25, lr=0.007, reg=0.05, n_epochs=20, seed=42):
        self.n_factors = n_factors
        self.lr = lr
        self.reg = reg
        self.n_epochs = n_epochs
        self.seed = seed

    def fit(self, df_train):
        rng = np.random.default_rng(self.seed)

        # Мапінг зовнішніх ID у внутрішні індекси
        unique_users = df_train['user_id'].unique()
        unique_items = df_train['item_id'].unique()
        self.user_map = {u: idx for idx, u in enumerate(unique_users)}
        self.item_map = {i: idx for idx, i in enumerate(unique_items)}

        n_users = len(unique_users)
        n_items = len(unique_items)

        # Глобальне середнє та ініціалізація зміщень і матриць прихованих факторів
        self.mu = float(df_train['rating'].mean())
        self.b_u = np.zeros(n_users, dtype=np.float64)
        self.b_i = np.zeros(n_items, dtype=np.float64)
        self.P = rng.normal(0.0, 0.1, size=(n_users, self.n_factors))
        self.Q = rng.normal(0.0, 0.1, size=(n_items, self.n_factors))

        u_indices = df_train['user_id'].map(self.user_map).values
        i_indices = df_train['item_id'].map(self.item_map).values
        ratings = df_train['rating'].values.astype(np.float64)

        # Навчання методом стохастичного градієнтного спуску (SGD)
        n_samples = len(ratings)
        for _ in range(self.n_epochs):
            order = rng.permutation(n_samples)
            for idx in order:
                u = u_indices[idx]
                i = i_indices[idx]
                r = ratings[idx]

                pred = self.mu + self.b_u[u] + self.b_i[i] + np.dot(self.P[u], self.Q[i])
                err = r - pred

                # Оновлення зміщень та векторів латентних ознак
                self.b_u[u] += self.lr * (err - self.reg * self.b_u[u])
                self.b_i[i] += self.lr * (err - self.reg * self.b_i[i])

                p_old = self.P[u].copy()
                self.P[u] += self.lr * (err * self.Q[i] - self.reg * self.P[u])
                self.Q[i] += self.lr * (err * p_old - self.reg * self.Q[i])

        return self

    def predict_one(self, uid, iid):
        has_u = uid in self.user_map
        has_i = iid in self.item_map

        # Обробка випадків "холодного старту"
        if not has_u and not has_i:
            est = self.mu
        elif not has_u:
            est = self.mu + self.b_i[self.item_map[iid]]
        elif not has_i:
            est = self.mu + self.b_u[self.user_map[uid]]
        else:
            u = self.user_map[uid]
            i = self.item_map[iid]
            est = self.mu + self.b_u[u] + self.b_i[i] + np.dot(self.P[u], self.Q[i])

        return float(np.clip(est, 1.0, 5.0))

    def recommend_top_n(self, uid, df_train, top_n=10):
        if uid not in self.user_map:
            return pd.DataFrame(columns=['item_id', 'predicted_rating'])

        seen_items = set(df_train.loc[df_train['user_id'] == uid, 'item_id'])
        candidates = [iid for iid in self.item_map if iid not in seen_items]

        scored = [(iid, self.predict_one(uid, iid)) for iid in candidates]
        recs = (
            pd.DataFrame(scored, columns=['item_id', 'predicted_rating'])
            .sort_values(by='predicted_rating', ascending=False)
            .head(top_n)
            .reset_index(drop=True)
        )
        return recs


# Навчання власної моделі на train_df
scratch_svd = BiasedSVDFromScratch(n_factors=25, lr=0.007, reg=0.05, n_epochs=20, seed=42)
scratch_svd.fit(train_df)

# Оцінка якості на незалежній тестовій вибірці test_df
actual_ratings = test_df['rating'].values
predicted_ratings = np.array([
    scratch_svd.predict_one(row.user_id, row.item_id)
    for row in test_df.itertuples(index=False)
])

scratch_rmse = np.sqrt(np.mean((actual_ratings - predicted_ratings) ** 2))
scratch_mae = np.mean(np.abs(actual_ratings - predicted_ratings))

print("Якість власної реалізації Biased SVD (SGD) на тесті:")
print(f"Test RMSE: {scratch_rmse:.6f}")
print(f"Test MAE:  {scratch_mae:.6f}")

# Формування Топ-10 рекомендацій для користувача '196'
target_uid = '196'
top10_scratch = scratch_svd.recommend_top_n(target_uid, train_df, top_n=10)
print(f"\nТоп-10 рекомендацій власної моделі для користувача {target_uid}:")
display(top10_scratch)

Якість власної реалізації Biased SVD (SGD) на тесті:
Test RMSE: 0.928993
Test MAE:  0.732498

Топ-10 рекомендацій власної моделі для користувача 196:


,item_id,predicted_rating
0,320,4.475740
1,318,4.392978
2,483,4.350602
3,114,4.324624
4,963,4.313766
5,408,4.277573
6,12,4.274450
7,64,4.272280
8,98,4.259766
9,427,4.254486
